# 01b — EDA General y Modelos Frecuentistas con Todos los Features

El notebook `01_eda_frecuentista.ipynb` sigue el ejemplo de Agresti (2002, cap. 4)
usando solo `W` como predictor. Aquí exploramos los cuatro features disponibles
(`W`, `Wt`, `C`, `S`) y ajustamos modelos frecuentistas completos para evaluar
si los features adicionales aportan poder predictivo sobre `Sa`.

**Features:**
- `W`: ancho del caparazón (cm) — continua
- `Wt`: peso (kg) — continua, correlacionada con W
- `C`: color (1=light medium, 2=medium, 3=dark medium, 4=dark) — ordinal
- `S`: condición de la espina (1=ambas buenas, 2=una dañada, 3=ambas dañadas) — ordinal
- `Sa`: número de satélites — variable respuesta (conteo)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style('whitegrid')
np.random.seed(42)

df = pd.read_csv('../data/agresti_crab_satellites_dataset.csv')
print(df.shape)
display(df.head())

## 1. Distribuciones univariadas

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Continuas
for ax, col in zip(axes[0], ['W', 'Wt', 'Sa']):
    ax.hist(df[col], bins=20, color='steelblue', edgecolor='white')
    ax.axvline(df[col].mean(), color='red', lw=1.5, linestyle='--', label=f'media={df[col].mean():.2f}')
    ax.set_title(col)
    ax.legend(fontsize=8)

# Categóricas
for ax, (col, labels) in zip(axes[1], [
    ('C', ['1-light\nmed', '2-med', '3-dark\nmed', '4-dark']),
    ('S', ['1-buenas', '2-una\ndañada', '3-ambas\ndañadas']),
]):
    counts = df[col].value_counts().sort_index()
    ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
    ax.set_xticks(counts.index)
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_title(col)

axes[1, 2].axis('off')
plt.suptitle('Distribuciones univariadas', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Correlaciones entre features

In [ ]:
corr = df[['W', 'Wt', 'C', 'S', 'Sa']].corr()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Heatmap
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=axes[0], square=True, linewidths=0.5)
axes[0].set_title('Correlación de Pearson')

# Correlaciones con Sa ordenadas
corr_sa = corr['Sa'].drop('Sa').sort_values(key=abs, ascending=False)
colors = ['steelblue' if v > 0 else 'tomato' for v in corr_sa]
axes[1].barh(corr_sa.index, corr_sa.values, color=colors)
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_xlabel('Correlación con Sa')
axes[1].set_title('Correlación con la variable respuesta')
for i, v in enumerate(corr_sa.values):
    axes[1].text(v + 0.01 * np.sign(v), i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 3. Relación de cada feature con Sa

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ['W', 'Wt']):
    ax.scatter(df[col], df['Sa'], alpha=0.3, color='steelblue')
    # LOWESS
    from statsmodels.nonparametric.smoothers_lowess import lowess
    smooth = lowess(df['Sa'], df[col], frac=0.5)
    ax.plot(smooth[:, 0], smooth[:, 1], color='red', lw=2, label='LOWESS')
    r, p = stats.pearsonr(df[col], df['Sa'])
    ax.set_title(f'Sa vs {col}  (r={r:.3f}, p={p:.3f})')
    ax.set_xlabel(col)
    ax.set_ylabel('Sa')
    ax.legend()

plt.suptitle('Relación entre features continuos y Sa', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

c_labels = {1: 'light med', 2: 'medium', 3: 'dark med', 4: 'dark'}
s_labels = {1: 'buenas', 2: 'una dañada', 3: 'ambas dañadas'}

for ax, (col, labels) in zip(axes, [('C', c_labels), ('S', s_labels)]):
    order = sorted(df[col].unique())
    sns.boxplot(x=col, y='Sa', data=df, order=order, ax=ax,
                palette='Blues', width=0.5)
    # Medias por grupo
    means = df.groupby(col)['Sa'].mean()
    ax.plot(range(len(order)), [means[k] for k in order],
            'ro--', ms=6, label='media')
    ax.set_xticklabels([labels[k] for k in order], fontsize=8)
    ax.set_title(f'Sa por {col}')
    ax.set_xlabel('')
    ax.legend()

plt.suptitle('Distribución de Sa por variable categórica', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
for col, name in [('C', 'Color'), ('S', 'Spine condition')]:
    summary = df.groupby(col)['Sa'].agg(
        n='count', mean='mean', var='var',
        prop_zeros=lambda x: (x == 0).mean()
    ).round(3)
    summary['disp_index'] = (summary['var'] / summary['mean']).round(3)
    print(f'\n=== Sa por {name} ===')
    display(summary)

## 4. Modelos frecuentistas con todos los features

`C` y `S` se tratan como continuas ordinales — un solo coeficiente por variable.
Si hubiera efectos no lineales habría que usar `C(C)` para dummy encoding.

`W` y `Wt` están altamente correlacionadas — ajustamos modelos con y sin `Wt`
para evaluar la multicolinealidad.

In [ ]:
for col in ['W', 'Wt', 'C', 'S']:
    df[f'{col}_sc'] = (df[col] - df[col].mean()) / df[col].std()

print('Correlación W_sc vs Wt_sc:', df['W_sc'].corr(df['Wt_sc']).round(3))

In [ ]:
poisson_specs = {
    'P1 — W':          'Sa ~ W_sc',
    'P2 — W + Wt':     'Sa ~ W_sc + Wt_sc',
    'P3 — W + C + S':  'Sa ~ W_sc + C_sc + S_sc',
    'P4 — todos':      'Sa ~ W_sc + Wt_sc + C_sc + S_sc',
}

poisson_results = {}
print(f"{'Modelo':<20} {'AIC':>8} {'BIC':>8} {'LogLik':>10}")
print('-' * 50)
for name, formula in poisson_specs.items():
    r = smf.poisson(formula=formula, data=df).fit(disp=0)
    poisson_results[name] = r
    print(f'{name:<20} {r.aic:>8.1f} {r.bic:>8.1f} {r.llf:>10.1f}')

In [ ]:
nb_specs = {
    'NB1 — W':         'Sa ~ W_sc',
    'NB2 — W + Wt':    'Sa ~ W_sc + Wt_sc',
    'NB3 — W + C + S': 'Sa ~ W_sc + C_sc + S_sc',
    'NB4 — todos':     'Sa ~ W_sc + Wt_sc + C_sc + S_sc',
}

nb_results = {}
print(f"{'Modelo':<22} {'AIC':>8} {'BIC':>8} {'LogLik':>10} {'alpha':>8}")
print('-' * 60)
for name, formula in nb_specs.items():
    r = smf.negativebinomial(formula=formula, data=df).fit(disp=0)
    nb_results[name] = r
    print(f'{name:<22} {r.aic:>8.1f} {r.bic:>8.1f} {r.llf:>10.1f} {r.params["alpha"]:>8.3f}')

In [ ]:
# Coeficientes del modelo completo Poisson vs NegBin
p_full = poisson_results['P4 — todos']
nb_full = nb_results['NB4 — todos']

coef_names = ['Intercept', 'W_sc', 'Wt_sc', 'C_sc', 'S_sc']

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, (res, title) in zip(axes, [(p_full, 'Poisson — modelo completo'),
                                    (nb_full, 'NegBin — modelo completo')]):
    coefs = res.params[coef_names]
    cis   = res.conf_int().loc[coef_names]
    y_pos = range(len(coef_names))
    ax.errorbar(coefs, y_pos,
                xerr=[coefs - cis[0], cis[1] - coefs],
                fmt='o', color='steelblue', capsize=4)
    ax.axvline(0, color='red', lw=1, linestyle='--')
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(coef_names)
    ax.set_xlabel('Coeficiente (escala log)')
    ax.set_title(title)

plt.suptitle('Coeficientes con IC 95% — modelo completo', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for name, r in poisson_results.items():
    rows.append({'Familia': 'Poisson', 'Modelo': name,
                 'AIC': round(r.aic, 1), 'BIC': round(r.bic, 1), 'LogLik': round(r.llf, 1)})
for name, r in nb_results.items():
    rows.append({'Familia': 'NegBin', 'Modelo': name,
                 'AIC': round(r.aic, 1), 'BIC': round(r.bic, 1), 'LogLik': round(r.llf, 1)})

comparison_df = pd.DataFrame(rows)
best_aic = comparison_df['AIC'].min()
comparison_df['ΔAIC'] = (comparison_df['AIC'] - best_aic).round(1)
display(comparison_df.sort_values('AIC'))

## 5. Conclusiones del EDA extendido

### Relaciones con Sa

- **W y Wt** tienen relación directa (positiva) con Sa y están altamente correlacionadas
entre sí. Incluir ambas en el mismo modelo introduce multicolinealidad sin ganancia
real. 

— Wt no aporta información más allá de la que ya captura W.

- **C y S** tienen relación inversa (negativa) con Sa: hembras de color más oscuro y con
peor condición de espina tienden a tener menos satélites. Las correlaciones son
más débiles que las de W/Wt.

### Comparabilidad entre familias (Poisson vs NegBin)

AIC y BIC **sí son comparables entre Poisson y NegBin** — ambos se derivan del mismo
LogLik sobre los mismos datos, y el parámetro extra de NegBin (alpha) ya está
contabilizado en la penalización. Todos los modelos NegBin superan a sus equivalentes
Poisson en AIC, BIC y LogLik, confirmando la sobredispersión identificada en el EDA base.

### Selección entre modelos NegBin

Las diferencias de AIC entre modelos NegBin son pequeñas — cualquiera podría
considerarse razonable. El BIC, que penaliza la complejidad más fuertemente
(log(n)·k vs 2·k de AIC, con n=173), muestra diferencias más claras:

- **Mejor modelo por BIC: NB1 (solo W)** — los features adicionales no compensan
el costo de complejidad.

- **Entre los modelos con features extra:** NB3 (W + C + S) supera a NB4 (todos),
confirmando que Wt es redundante dado W y añade ruido por multicolinealidad.

- **Conclusión práctica:** para este dataset y siguiendo parsimonia, el modelo con
solo W es suficiente — consistente con la elección de Agresti (2002)."

---

## ¿Qué sigue?

El análisis frecuentista confirma que **W (ancho de caparazón)** es el predictor
más relevante y que el modelo parsimonioso (`Sa ~ W`, NegBin) es preferible por BIC.
Sin embargo, los modelos ajustados por MLE producen estimaciones puntuales:
no cuantifican incertidumbre paramétrica ni permiten incorporar conocimiento previo.

En el **notebook 03** transitamos al paradigma bayesiano. Comenzamos desde el método
más elemental (estimación por grid) y escalamos progresivamente hasta Stan — para que
la herramienta no oculte la idea.